In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

print("OpsMind AI - Model Evaluation & Explainability")
print("Libraries Imported Successfully!")

OpsMind AI - Model Evaluation & Explainability
Libraries Imported Successfully!


In [2]:
model_files = [
    "../data/processed/best_log_anomaly_model.pkl",
    "../data/processed/best_hdfs_anomaly_model.pkl"
]

for file in model_files:
    print(file, "->", os.path.exists(file))

../data/processed/best_log_anomaly_model.pkl -> True
../data/processed/best_hdfs_anomaly_model.pkl -> True


In [3]:
# Check all saved model artifacts
processed_path = "../data/processed"

model_files = [
    file for file in os.listdir(processed_path)
    if file.endswith(".pkl")
]

print("SAVED MODEL FILES:")
for file in model_files:
    print("-", file)

print("\nTotal Model Files:", len(model_files))

SAVED MODEL FILES:
- best_hdfs_anomaly_model.pkl
- best_log_anomaly_model.pkl
- best_ticket_classifier.pkl

Total Model Files: 3


In [4]:
# Load saved model packages
log_package = joblib.load(
    "../data/processed/best_log_anomaly_model.pkl"
)

hdfs_package = joblib.load(
    "../data/processed/best_hdfs_anomaly_model.pkl"
)

ticket_package = joblib.load(
    "../data/processed/best_ticket_classifier.pkl"
)

# Inspect model package structure
print("OPENSTACK MODEL:")
print(type(log_package))
print(log_package.keys() if isinstance(log_package, dict) else "Direct Model")

print("\nHDFS MODEL:")
print(type(hdfs_package))
print(hdfs_package.keys() if isinstance(hdfs_package, dict) else "Direct Model")

print("\nTICKET CLASSIFIER:")
print(type(ticket_package))
print(ticket_package.keys() if isinstance(ticket_package, dict) else "Direct Model")

OPENSTACK MODEL:
<class 'dict'>
dict_keys(['model', 'vectorizer', 'threshold', 'model_name', 'target_classes'])

HDFS MODEL:
<class 'dict'>
dict_keys(['model', 'vectorizer', 'model_name'])

TICKET CLASSIFIER:
<class 'sklearn.pipeline.Pipeline'>
Direct Model


In [5]:
# Inspect components stored inside each model package

packages = {
    "OpenStack": log_package,
    "HDFS": hdfs_package,
    "Ticket Classifier": ticket_package
}

for name, package in packages.items():
    print("\n", "=" * 50)
    print(name.upper())

    if isinstance(package, dict):
        for key, value in package.items():
            print(f"{key}: {type(value).__name__}")
    else:
        print("Model Type:", type(package).__name__)


OPENSTACK
model: LogisticRegression
vectorizer: TfidfVectorizer
threshold: float64
model_name: str
target_classes: dict

HDFS
model: RandomForestClassifier
vectorizer: TfidfVectorizer
model_name: str

TICKET CLASSIFIER
Model Type: Pipeline


In [6]:
# OpenStack details
print("OPENSTACK")
print("Model:", log_package["model_name"])
print("Threshold:", log_package["threshold"])
print("Target Classes:", log_package["target_classes"])

# HDFS details
print("\nHDFS")
print("Model:", hdfs_package["model_name"])

# Ticket classifier details
print("\nTICKET CLASSIFIER")
print("Pipeline Steps:")
for name, step in ticket_package.steps:
    print(name, "->", type(step).__name__)

OPENSTACK
Model: Logistic Regression
Threshold: 0.9500736046886544
Target Classes: {0: 'Normal', 1: 'Abnormal'}

HDFS
Model: Random Forest

TICKET CLASSIFIER
Pipeline Steps:
tfidf -> TfidfVectorizer
classifier -> LinearSVC


In [7]:
# Function to extract top important features
def show_top_features(model, vectorizer, model_name, top_n=15):

    feature_names = vectorizer.get_feature_names_out()

    if hasattr(model, "feature_importances_"):
        importance = model.feature_importances_

    elif hasattr(model, "coef_"):
        importance = np.abs(model.coef_).mean(axis=0)

    else:
        print(model_name, "- Feature importance not available")
        return

    top_indices = np.argsort(importance)[-top_n:][::-1]

    result = pd.DataFrame({
        "Feature": feature_names[top_indices],
        "Importance": importance[top_indices]
    })

    print("\n", model_name, "- TOP IMPORTANT FEATURES")
    display(result)


# OpenStack
show_top_features(
    log_package["model"],
    log_package["vectorizer"],
    "OpenStack"
)

# HDFS
show_top_features(
    hdfs_package["model"],
    hdfs_package["vectorizer"],
    "HDFS"
)

# Ticket Classification
ticket_model = ticket_package.named_steps["classifier"]
ticket_vectorizer = ticket_package.named_steps["tfidf"]

show_top_features(
    ticket_model,
    ticket_vectorizer,
    "ITSM Ticket Classifier"
)


 OpenStack - TOP IMPORTANT FEATURES


,Feature,Importance
0,11 12,17.044525
1,11 11,16.442907
2,12,12.946450
3,4431,6.509138
4,4544,5.962641
5,4727,5.781144
6,a694,5.742557
7,46bb,5.705306
8,9c0a,5.517695
9,4ee1,5.438698



 HDFS - TOP IMPORTANT FEATURES


,Feature,Importance
0,e26,0.191664
1,e9,0.161663
2,e11,0.161147
3,e23,0.082492
4,e3,0.072516
5,e5,0.064943
6,e4,0.056387
7,e21,0.050227
8,e20,0.041959
9,e22,0.036267



 ITSM Ticket Classifier - TOP IMPORTANT FEATURES


,Feature,Importance
0,oracle,1.142530
1,sent,0.893093
2,windows,0.682443
3,access,0.666802
4,laptop,0.664276
5,manager,0.640605
6,need,0.630910
7,pas,0.624961
8,hello,0.601881
9,network,0.586898


In [8]:
# Extract and save top 15 features for each model

def get_top_features(model, vectorizer, model_name, top_n=15):

    feature_names = vectorizer.get_feature_names_out()

    if hasattr(model, "feature_importances_"):
        importance = model.feature_importances_
    elif hasattr(model, "coef_"):
        importance = np.abs(model.coef_).mean(axis=0)
    else:
        print(model_name, "importance unavailable")
        return None

    indices = np.argsort(importance)[-top_n:][::-1]

    return pd.DataFrame({
        "Feature": feature_names[indices],
        "Importance": importance[indices]
    })


# Extract features
openstack_features = get_top_features(
    log_package["model"], log_package["vectorizer"], "OpenStack"
)

hdfs_features = get_top_features(
    hdfs_package["model"], hdfs_package["vectorizer"], "HDFS"
)

ticket_features = get_top_features(
    ticket_package.named_steps["classifier"],
    ticket_package.named_steps["tfidf"],
    "ITSM"
)

# Save results
if openstack_features is not None:
    openstack_features.to_csv(
        "../data/processed/openstack_feature_importance.csv", index=False
    )

if hdfs_features is not None:
    hdfs_features.to_csv(
        "../data/processed/hdfs_feature_importance.csv", index=False
    )

if ticket_features is not None:
    ticket_features.to_csv(
        "../data/processed/ticket_feature_importance.csv", index=False
    )

print("Feature Importance Reports Saved Successfully!")

Feature Importance Reports Saved Successfully!


In [9]:
# Final OpsMind AI Model Summary

model_summary = pd.DataFrame([
    {
        "Dataset": "OpenStack Logs",
        "Selected Model": log_package["model_name"],
        "Model Type": "Supervised Classification"
    },
    {
        "Dataset": "HDFS Logs",
        "Selected Model": hdfs_package["model_name"],
        "Model Type": "Supervised Classification"
    },
    {
        "Dataset": "ITSM Tickets",
        "Selected Model": "Linear SVM",
        "Model Type": "Supervised Text Classification"
    }
])

display(model_summary)

# Save model summary
model_summary.to_csv(
    "../data/processed/final_model_summary.csv",
    index=False
)

print("Final Model Summary Saved Successfully!")

,Dataset,Selected Model,Model Type
0,OpenStack Logs,Logistic Regression,Supervised Classification
1,HDFS Logs,Random Forest,Supervised Classification
2,ITSM Tickets,Linear SVM,Supervised Text Classification


Final Model Summary Saved Successfully!


# Notebook 10 – Model Evaluation & Explainability

## Objective
Evaluate saved machine learning models and inspect their important features.

## Selected Models
1. OpenStack Log Anomaly Detection – Logistic Regression
2. HDFS Log Anomaly Detection – Random Forest
3. ITSM Ticket Classification – TF-IDF + Linear SVM

## Explainability
- Extracted important features from trained models.
- Saved feature importance reports.
- Inspected model configurations and saved artifacts.

## Important Limitations
- OpenStack and HDFS are independent datasets.
- Their evaluation metrics are not directly comparable.
- Further validation on unseen log patterns is required.
- Feature importance indicates model associations, not causation.

## Conclusion
Model artifacts were loaded and inspected successfully.
Feature importance and model summary reports were prepared
for further integration into the OpsMind AI system.